## Configuración del Entorno

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/reproduccion_gan_fraud

!python -m pip install -q -r requirements.txt

/content/drive/MyDrive/reproduccion_gan_fraud


In [3]:
import joblib
import numpy as np
import pandas as pd
import sklearn
import imblearn
import torch
import yaml

print("Dependencias importadas correctamente.")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

Dependencias importadas correctamente.
PyTorch: 2.11.0+cu128
CUDA disponible: True


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/reproduccion_gan_fraud")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la ruta: {PROJECT_ROOT}")

project_path = str(PROJECT_ROOT)

if project_path not in sys.path:
    sys.path.insert(0, project_path)

print("Proyecto agregado al PYTHONPATH:")
print(PROJECT_ROOT)


Proyecto agregado al PYTHONPATH:
/content/drive/MyDrive/reproduccion_gan_fraud


In [5]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment_config.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Configuración cargada correctamente.")
print("Proyecto:", config["project"]["name"])
print("Semilla:", config["project"]["seed"])

Configuración cargada correctamente.
Proyecto: reproduccion_gan_fraud
Semilla: 42


In [6]:
from src import data_pipeline
from src import classifier_pipeline
from src import evaluation
from src import gan_pipeline
from src import sample_generation
from src import augmentation_experiment

print("Todos los módulos se importaron correctamente.")

Todos los módulos se importaron correctamente.


## Procesamiento del dataset

In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

processed_dir = PROJECT_ROOT / config["paths"]["data"]["processed"]
phase1_dir = PROJECT_ROOT / config["paths"]["artifacts"]["phase1_data"]
raw_csv = PROJECT_ROOT / config["paths"]["data"]["raw_csv"]

paths = data_pipeline.build_persistence_paths(processed_dir, phase1_dir)

if all(path.exists() for path in vars(paths).values()):
    prepared = data_pipeline.load_prepared_data(processed_dir, phase1_dir)
    print("Artefactos de Fase 1 cargados.")
else:
    prepared = data_pipeline.prepare_data(raw_csv, config)
    data_pipeline.persist_prepared_data(
        prepared,
        processed_dir,
        phase1_dir,
        overwrite=False,
    )
    print("Fase 1 ejecutada y persistida.")

X_train, y_train = prepared.X_train, prepared.y_train
X_test, y_test = prepared.X_test, prepared.y_test

print("Train:", X_train.shape, np.bincount(y_train))
print("Test:", X_test.shape, np.bincount(y_test))
print("GPU:", torch.cuda.is_available())

Fase 1 ejecutada y persistida.
Train: (170236, 30) [169921    315]
Test: (113490, 30) [113332    158]
GPU: True


## Modelo Baseline

In [8]:
phase2_dir = PROJECT_ROOT / config["paths"]["artifacts"]["phase2_baseline"]
phase2_dir.mkdir(parents=True, exist_ok=True)

baseline_result = classifier_pipeline.run_classifier_pipeline(
    X_train,
    y_train,
    config,
    device="auto",
    verbose=True,
)

test_loader = classifier_pipeline.create_dataloader(
    X_test,
    y_test,
    batch_size=config["classifier_training"]["batch_size"],
    shuffle=False,
    seed=config["project"]["seed"],
    num_workers=config["data_loader"]["num_workers"],
    pin_memory=config["data_loader"]["pin_memory"],
    device="auto",
)

baseline_metrics = evaluation.evaluate_model(
    baseline_result["final_model"],
    test_loader,
    device="auto",
    include_probability_diagnostics=True,
)

paper_metrics = {
    "sensitivity": 0.70229,
    "specificity": 0.99998,
    "precision": 0.97872,
    "f1": 0.81778,
    "accuracy": 0.99964,
}

paper_comparison = evaluation.compare_with_reference(
    baseline_metrics,
    paper_metrics,
)

torch.save(
    {
        "model_state_dict": baseline_result["final_model"].state_dict(),
        "best_epoch": baseline_result["best_epoch"],
        "model_config": config["classifier_model"],
    },
    phase2_dir / "baseline_classifier.pt",
)

pd.DataFrame(baseline_result["selection_history"]).to_csv(
    phase2_dir / "baseline_validation_history.csv", index=False
)
pd.DataFrame(baseline_result["final_history"]).to_csv(
    phase2_dir / "baseline_final_training_history.csv", index=False
)
pd.DataFrame(paper_comparison).to_csv(
    phase2_dir / "baseline_paper_comparison.csv", index=False
)

with (phase2_dir / "baseline_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(baseline_metrics, file, indent=4)

print("Mejor época:", baseline_result["best_epoch"])
print("Métricas baseline:", baseline_metrics)

Epoch 001 | loss=0.159131 | PR-AUC=0.060328 | ROC-AUC=0.777954 | recall=0.0000 | f1=0.0000 | pred+=0 | 2.2s
Epoch 002 | loss=0.026570 | PR-AUC=0.110086 | ROC-AUC=0.806445 | recall=0.0000 | f1=0.0000 | pred+=0 | 2.5s
Epoch 003 | loss=0.018829 | PR-AUC=0.141522 | ROC-AUC=0.819993 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.9s
Epoch 004 | loss=0.016105 | PR-AUC=0.173327 | ROC-AUC=0.829091 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.4s
Epoch 005 | loss=0.014812 | PR-AUC=0.192372 | ROC-AUC=0.835963 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.3s
Epoch 006 | loss=0.014105 | PR-AUC=0.212064 | ROC-AUC=0.842119 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.4s
Epoch 007 | loss=0.013695 | PR-AUC=0.226911 | ROC-AUC=0.847927 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.4s
Epoch 008 | loss=0.013456 | PR-AUC=0.236474 | ROC-AUC=0.854167 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.4s
Epoch 009 | loss=0.013319 | PR-AUC=0.251970 | ROC-AUC=0.862426 | recall=0.0000 | f1=0.0000 | pred+=0 | 1.3s
Epoch 010 | loss=0.013258 | 

# GAN Entrenamiento

In [9]:
# Segundo ajuste experimental de la GAN: solo modifica config en memoria.
config["gan_training"].update({
    "batch_size": 32,
    "epochs": 500,
    "generator_learning_rate":  1e-3,
    "discriminator_learning_rate": 5e-4,
    "discriminator_steps": 1,
    "generator_steps": 1,
    "diagnostic_samples": 64,
    "log_frequency": 25,
})

# Inicialización intermedia: suficiente variación sin saturar las Sigmoid.
config["gan_training"]["initialization"] = {
    "method": "uniform",
    "min_value": -0.1,
    "max_value": 0.1,
}

print("Configuración GAN del segundo ensayo:")
print("Batch size:", config["gan_training"]["batch_size"])
print("Épocas:", config["gan_training"]["epochs"])
print("LR generador:", config["gan_training"]["generator_learning_rate"])
print("LR discriminador:", config["gan_training"]["discriminator_learning_rate"])
print("Inicialización:", config["gan_training"]["initialization"])


Configuración GAN del segundo ensayo:
Batch size: 32
Épocas: 500
LR generador: 0.001
LR discriminador: 0.0005
Inicialización: {'method': 'uniform', 'min_value': -0.1, 'max_value': 0.1}


In [10]:
phase3_dir = PROJECT_ROOT / config["paths"]["artifacts"]["phase3_gan"]
phase3_dir.mkdir(parents=True, exist_ok=True)

gan_result = gan_pipeline.run_gan_pipeline(
    X_train,
    y_train,
    config,
    device="auto",
    verbose=True,
)

torch.save(
    {
        "model_state_dict": gan_result["generator"].state_dict(),
        "model_config": config["gan_model"]["generator"],
        "noise_dim": config["gan_model"]["noise_dim"],
        "data_dim": config["gan_model"]["data_dim"],
    },
    phase3_dir / "gan_generator.pt",
)

torch.save(
    {
        "model_state_dict": gan_result["discriminator"].state_dict(),
        "model_config": config["gan_model"]["discriminator"],
    },
    phase3_dir / "gan_discriminator.pt",
)

pd.DataFrame(gan_result["history"]).to_csv(
    phase3_dir / "gan_training_history.csv", index=False
)

with (phase3_dir / "gan_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "metadata": gan_result["metadata"],
            "preview_diagnostics": gan_result["preview_diagnostics"],
            "termination_reason": gan_result["termination_reason"],
        },
        file,
        indent=4,
    )

np.savez_compressed(
    phase3_dir / "synthetic_samples_preview.npz",
    samples=gan_result["preview_samples"],
)

print("GAN entrenada.")
print(gan_result["metadata"])
print(gan_result["preview_diagnostics"])

Epoch 0025/0500 | D=1.386208 | G=0.693512 | D(real)=0.4999 | D(fake)=0.4998 | 0.05s
Epoch 0050/0500 | D=1.385308 | G=0.693535 | D(real)=0.5003 | D(fake)=0.4998 | 0.04s
Epoch 0075/0500 | D=1.384129 | G=0.693949 | D(real)=0.5007 | D(fake)=0.4996 | 0.05s
Epoch 0100/0500 | D=1.382237 | G=0.694749 | D(real)=0.5012 | D(fake)=0.4992 | 0.05s
Epoch 0125/0500 | D=1.378692 | G=0.696361 | D(real)=0.5022 | D(fake)=0.4984 | 0.05s
Epoch 0150/0500 | D=1.371079 | G=0.699986 | D(real)=0.5042 | D(fake)=0.4966 | 0.05s
Epoch 0175/0500 | D=1.353049 | G=0.708930 | D(real)=0.5090 | D(fake)=0.4922 | 0.05s
Epoch 0200/0500 | D=1.319486 | G=0.725367 | D(real)=0.5180 | D(fake)=0.4841 | 0.06s
Epoch 0225/0500 | D=1.498392 | G=0.643595 | D(real)=0.4714 | D(fake)=0.5258 | 0.04s
Epoch 0250/0500 | D=1.407602 | G=0.684593 | D(real)=0.4938 | D(fake)=0.5044 | 0.05s
Epoch 0275/0500 | D=1.344433 | G=0.714002 | D(real)=0.5108 | D(fake)=0.4896 | 0.05s
Epoch 0300/0500 | D=1.282181 | G=0.747570 | D(real)=0.5280 | D(fake)=0.4744 

# Experimento: Generación $N_{g}$ muestras artificales

In [11]:
phase4_dir = PROJECT_ROOT / config["paths"]["artifacts"]["phase4_augmentation"]
histories_dir = phase4_dir / "histories"
phase4_dir.mkdir(parents=True, exist_ok=True)
histories_dir.mkdir(parents=True, exist_ok=True)

results_path = phase4_dir / "combined_results.csv"

if results_path.exists():
    saved_rows = pd.read_csv(results_path).to_dict("records")
    completed = {
        (str(row["method"]), int(row["generated_samples"]), int(row["seed"]))
        for row in saved_rows
        if row.get("status") == "completed"
    }
else:
    saved_rows = []
    completed = set()

def save_experiment(payload):
    row = dict(payload["row"])

    saved_rows[:] = [
        existing for existing in saved_rows
        if not (
            str(existing["method"]) == str(row["method"])
            and int(existing["generated_samples"]) == int(row["generated_samples"])
            and int(existing["seed"]) == int(row["seed"])
        )
    ]
    saved_rows.append(row)
    pd.DataFrame(saved_rows).to_csv(results_path, index=False)

    history = payload.get("history")
    if history:
        name = f'{row["method"]}_{int(row["generated_samples"])}'
        pd.DataFrame(history).to_csv(
            histories_dir / f"{name}_history.csv",
            index=False,
        )

augmentation_result = augmentation_experiment.run_augmentation_experiments(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    config=config,
    baseline_epochs=baseline_result["best_epoch"],
    generator=gan_result["generator"],
    baseline_metrics=baseline_metrics,
    device="auto",
    include_baseline=True,
    keep_histories=False,
    keep_models=False,
    completed_experiments=completed,
    on_experiment_end=save_experiment,
    verbose=True,
)

results_df = pd.DataFrame(saved_rows).sort_values(
    ["method", "generated_samples"]
)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        [
            "method",
            "generated_samples",
            "status",
            "sensitivity",
            "specificity",
            "precision",
            "f1",
            "accuracy",
            "fp",
            "fn",
        ]
    ]
)

print("Experimentos completos:", len(results_df))
print("Resultados guardados en:", results_path)

Epoch 001/100 | loss=0.147816 | momentum=0.5000 | 1.3s
Epoch 002/100 | loss=0.026691 | momentum=0.5544 | 1.7s
Epoch 003/100 | loss=0.020182 | momentum=0.6089 | 2.4s
Epoch 004/100 | loss=0.017988 | momentum=0.6633 | 1.6s
Epoch 005/100 | loss=0.017001 | momentum=0.7178 | 1.3s
Epoch 006/100 | loss=0.016495 | momentum=0.7722 | 1.3s
Epoch 007/100 | loss=0.016227 | momentum=0.8267 | 1.3s
Epoch 008/100 | loss=0.016079 | momentum=0.8811 | 1.3s
Epoch 009/100 | loss=0.016001 | momentum=0.9356 | 1.3s
Epoch 010/100 | loss=0.015940 | momentum=0.9900 | 1.3s
Epoch 011/100 | loss=0.015850 | momentum=0.9900 | 1.4s
Epoch 012/100 | loss=0.015733 | momentum=0.9900 | 2.4s
Epoch 013/100 | loss=0.015622 | momentum=0.9900 | 2.0s
Epoch 014/100 | loss=0.015487 | momentum=0.9900 | 1.3s
Epoch 015/100 | loss=0.015342 | momentum=0.9900 | 1.3s
Epoch 016/100 | loss=0.015183 | momentum=0.9900 | 1.3s
Epoch 017/100 | loss=0.015002 | momentum=0.9900 | 1.3s
Epoch 018/100 | loss=0.014798 | momentum=0.9900 | 1.3s
Epoch 019/

,method,generated_samples,status,sensitivity,specificity,precision,f1,accuracy,fp,fn
0,baseline,0,completed,0.715190,0.999788,0.824818,0.766102,0.999392,24,45
1,gan,79,completed,0.740506,0.999771,0.818182,0.777409,0.999410,26,41
2,gan,158,completed,0.740506,0.999771,0.818182,0.777409,0.999410,26,41
3,gan,315,completed,0.746835,0.999762,0.813793,0.778878,0.999410,27,40
4,gan,630,completed,0.696203,0.999797,0.827068,0.756014,0.999374,23,48
5,gan,945,completed,0.746835,0.999753,0.808219,0.776316,0.999401,28,40
6,gan,1260,completed,0.753165,0.999735,0.798658,0.775244,0.999392,30,39
7,gan,2520,completed,0.753165,0.999744,0.804054,0.777778,0.999401,29,39
8,gan,3150,completed,0.727848,0.999771,0.815603,0.769231,0.999392,26,43
9,gan,6300,completed,0.746835,0.999753,0.808219,0.776316,0.999401,28,40


Experimentos completos: 21
Resultados guardados en: /content/drive/MyDrive/reproduccion_gan_fraud/artifacts/phase4_augmentation/combined_results.csv
